# Finding a Downed RC Plane in 3,000 Aerial Images

This project was born from a Reddit post. Someone lost a turbine RC plane over the desert, flew an aerial mapping grid over the search area, and ended up with ~3,000 high-res images to comb through by hand:

> **[Need Help] Searching 3,000 aerial images to locate a downed turbine RC plane**
>
> Hey guys. I recently lost a turbine RC plane over the desert and ran an aerial mapping grid over the search area. I now have about 3,000 high-res images to comb through and was thinking that maybe someone knew how to cook up a quick cv script that could flag stuff for manual review.
>
> I built the plane myself and it would be sick to find it. I'll paypal/cashapp/zelle etc $50 bucks to whoever finds it lol
>
> Note: one of the images in the .zip is renamed as "0000", it's a false positive that has a shadow on the top left that looks EXACTLY like the plane... (I went and checked it out irl)

Manually reviewing thousands of near-identical desert tiles is exactly the needle-in-a-haystack problem [FiftyOne](https://docs.voxel51.com) is built for. This notebook walks through:

1. Loading the aerial survey into FiftyOne
2. Adding CLIP embeddings for semantic search and a 2D visualization
3. Running the **Plane Finder** plugin to flag plane-like objects
4. Building saved views that surface the best candidates for review

In [ ]:
#install prereqs

!pip install fiftyone umap-learn open-clip-torch

We'll make use of a plugin

## 1. Get the data

The raw search-grid images are available here:

**https://drive.google.com/drive/folders/1FJFQVgpgEg0lSm2f3-DRukAhsEYdcByD?usp=sharing**

Download and unzip them into a local `scans/` folder, then load the directory into FiftyOne as an image dataset and compute basic metadata (image width, height, etc.):

In [ ]:
import fiftyone as fo

dataset = fo.Dataset.from_dir(
    dataset_dir="scans",
    dataset_type=fo.types.ImageDirectory,
    name="Ariel_Scans",
    persistent=True,
    overwrite=True,
)

dataset.compute_metadata()

## 2. Add CLIP embeddings, similarity, and a 2D map

Embeddings let us search the dataset semantically and spot structure. We:

- run an **open-clip** model to get a feature vector per image (`clip_embeddings`),
- build a **similarity index** (`clip_sim`) so we can sort by visual similarity to any sample, and
- compute a **UMAP visualization** (`clip_viz`) — a 2D scatter you can lasso-select in the App's Embeddings panel to find clusters and outliers.

This is the natural-image complement to the geometry-based plugin we run later: embeddings find *visually* similar tiles, while the plugin finds *geometrically* plane-like blobs.

In [ ]:
import fiftyone.zoo as foz
import fiftyone.brain as fob

clip_model = foz.load_zoo_model("open-clip-torch")

dataset.compute_embeddings(
    model=clip_model,
    embeddings_field="clip_embeddings",
)

fob.compute_similarity(
    dataset,
    embeddings="clip_embeddings",
    brain_key="clip_sim",
)

fob.compute_visualization(
    dataset,
    embeddings="clip_embeddings",
    brain_key="clip_viz",
    method="umap",
)

## Shortcut: load the prepared dataset from Hugging Face

Everything above — images, metadata, embeddings, similarity, and the UMAP visualization — is already computed and published, so you can skip the setup entirely and pull the ready-to-go dataset straight from the Hugging Face Hub:

In [ ]:
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# Other available arguments include 'max_samples', 'persistent', etc.
dataset = load_from_hub("harpreetsahota/ariel_scans")

## 3. Flag plane-like objects with the Plane Finder plugin

The desert background is almost uniform mid-gray and the only dark features are scrubby vegetation, so tone alone won't separate the plane from a bush. The **Plane Finder** plugin (in this repo's `plane_finder/` folder) instead scores each dark blob on *shape and surface character*: an engineered carbon-fiber airframe is elongated, fills its bounding box, has a smooth low-variance interior, and is solid and convex. Vegetation is none of those.

The plugin ships two operators, both runnable from the Python SDK or the App:

- **`preview_dark_mask`** — a calibration helper. It writes the thresholded dark mask as a heatmap on a few samples so you can visually tune `dark_threshold` before a full run.
- **`find_plane`** — the detector. It writes a `plane_candidates` detections field (score in `confidence`, plus per-detection attributes like `rect_aspect` and `interior_std`) and a `max_plane_candidates_score` per sample for ranking.

To make the plugin available, put this repo's `plane_finder/` folder on your FiftyOne plugins path (symlink it into the folder printed by `fiftyone plugins dir`). Then load the operators and calibrate the threshold on a handful of images:

In [ ]:
import fiftyone.operators as foo

preview_dark_mask = foo.get_operator("@harpreetsahota/plane_finder/preview_dark_mask")
find_plane = foo.get_operator("@harpreetsahota/plane_finder/find_plane")

# Write the dark-mask heatmap on 9 images so we can eyeball the threshold in the App
await preview_dark_mask(dataset.take(9), num_samples=9, dark_threshold=105)

In [ ]:
# Run the detector across the dataset. Keep only confident candidates and cap how
# many boxes are written per image so the field stays clean. Flip on delegate=True
# to run it as a background job for the full 6,500-image set.
await find_plane(
    dataset,
    label_field="plane_candidates",
    dark_threshold=105,
    min_score=0.5,
    max_per_image=5,
)

### Running the plugin from the App

Both operators are also available interactively. Launch the App, open the operator browser with the backtick (`` ` ``) key, search for **"Find Plane"** or **"Preview Dark Mask"**, and fill in the form. Every input has an inline description, and you can flip on **Delegate execution** to run large jobs in the background (process them with `fiftyone delegated launch`).

![Plane Finder plugin in the FiftyOne App](./assets/plane_finder_demo.gif)

## 4. Build saved views for review

The operator output turns "scroll 3,000 images" into "review a ranked shortlist." We build a few **saved views** — they persist on the dataset, so anyone who loads it can reopen them in the App from the **Saved Views** dropdown:

- **`candidates_by_score`** — every sample with at least one candidate, ranked by its best score.
- **`high_confidence`** — only strong detections (`confidence > 0.6`), weak boxes dropped.
- **`carbon_like`** — the most airframe-like detections: elongated *and* smooth-surfaced.

In [ ]:
from fiftyone import ViewField as F

# A. Every sample with a candidate, ranked by its best score
candidates_by_score = dataset.match(
    F("max_plane_candidates_score") > 0
).sort_by("max_plane_candidates_score", reverse=True)

dataset.save_view("candidates_by_score", candidates_by_score, overwrite=True)

# B. High-confidence detections only (weak boxes filtered out)
high_confidence = dataset.filter_labels(
    "plane_candidates", F("confidence") > 0.6
).sort_by("max_plane_candidates_score", reverse=True)

dataset.save_view("high_confidence", high_confidence, overwrite=True)

# C. The most airframe-like: elongated AND smooth-surfaced
carbon_like = dataset.filter_labels(
    "plane_candidates",
    (F("rect_aspect") > 3.0) & (F("interior_std") < 13.0),
).sort_by("max_plane_candidates_score", reverse=True)

dataset.save_view("carbon_like", carbon_like, overwrite=True)

dataset.list_saved_views()

In [ ]:
session = fo.launch_app(dataset.load_saved_view("candidates_by_score"))

## Where to go from here

- Walk the top of `candidates_by_score` — the real airframe should surface near the top.
- Found a true positive? Use the `clip_sim` index with `sort_by_similarity` to pull visually similar tiles and check neighboring grid cells.
- The infamous `0000.jpeg` is the known shadow false-positive — a useful sanity check that both your eye and the score can tell it apart from the real thing.
- Tune `dark_threshold` and the size gates (`min_diag_px` / `max_diag_px`) to your imagery; see the plugin README for the full knob guide.